# sparse_knn — generation and scoring

---
## 1 — Host and working tree

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
import os

if not os.path.isdir('Style-Aware-MT') and not os.path.exists('manage.py'):
    !git clone https://github.com/prnamhr/Style-Aware-MT.git
if os.path.isdir('Style-Aware-MT'):
    %cd Style-Aware-MT
!git rev-parse --short HEAD

In [ ]:
%pip install -r requirements.txt

In [ ]:
# The pipeline is text-only and these three ship against a torch the pins contradict.
%pip uninstall -q -y torchvision torchaudio torchcodec

In [ ]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

---
## 2 — Run parameters and the matched-contrast gate

In [ ]:
import hashlib
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import yaml

PY = sys.executable
COMET_PY = '.venv-comet/bin/python'
SPLIT = 'val'
ARM, BASELINE = 'sparse_knn', 'knn_fewshot'
CONFIG, BASE_CONFIG = Path('configs/sparse_knn.yaml'), Path('configs/base_qwen.yaml')
OUT = Path('outputs')

DIAG = Path(f'results/sparse_selection_{SPLIT}.json')
ROUTING = Path(f'results/sparse_routing_{SPLIT}.json')

N_BOOT, N_STYLO, ALPHA, SEED = 10000, 2000, 0.05, 42

CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
RETR, SPA, RAR, PROMPT = CFG['retrieval'], CFG['sparse'], CFG['rarity'], CFG['prompt']
RARITY_PATH = Path(RAR['out'])
print(f"{ARM} against {BASELINE}: k={RETR['k']}, m={SPA['m']}, "
      f"df band [{RAR['df_min']}, {RAR['df_max']}], min_query_terms={SPA['min_query_terms']}")

In [ ]:
BASE = yaml.safe_load(BASE_CONFIG.read_text(encoding='utf-8'))
for block in ('generator', 'prompt', 'retrieval', 'data', 'output'):
    assert CFG[block] == BASE[block], f'{block} differs from {BASE_CONFIG}: {CFG[block]}'
assert set(CFG) - set(BASE) == {'rarity', 'sparse'}, sorted(set(CFG) - set(BASE))
assert set(BASE) - set(CFG) == {'afsp'}, sorted(set(BASE) - set(CFG))
print(f'{CONFIG.name} differs from {BASE_CONFIG.name} in the selection blocks only')

In [ ]:
VAL = [json.loads(x) for x in Path(CFG['data']['eval_file']).open(encoding='utf-8') if x.strip()]
SRC = [r['input'] for r in VAL]

BASE_ROWS = [json.loads(x) for x in (OUT / f'{BASELINE}_{SPLIT}.jsonl').open(encoding='utf-8')
             if x.strip()]
assert len(BASE_ROWS) == len(VAL), f'{BASELINE}: {len(BASE_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in BASE_ROWS] == SRC, f'{BASELINE} is not aligned to {SPLIT}.jsonl'
assert all(r['model'] == CFG['generator']['model'] for r in BASE_ROWS), 'a different base model'
print(f'{len(VAL)} {SPLIT} segments; {BASELINE} present and aligned on {BASE_ROWS[0]["model"]}')

---
## 3 — The rarity list and the index

In [ ]:
RARITY = json.loads(RARITY_PATH.read_text(encoding='utf-8'))
RARITY_SHA = hashlib.sha256(RARITY_PATH.read_bytes()).hexdigest()

assert RARITY['config']['df_min'] == RAR['df_min'], RARITY['config']
assert RARITY['config']['df_max'] == RAR['df_max'], RARITY['config']
assert RARITY['df_histogram']['irregular']['1'] == 0, 'hapaxes are back in the list'
assert RARITY['df_observed'] == [RAR['df_min'], RAR['df_max']], RARITY['df_observed']
print(f"{RARITY['n_irregular']} irregular terms, df {RARITY['df_observed']}, "
      f"{RARITY['selected_frac']:.1%} of the vocabulary")
print(f'sha256 {RARITY_SHA[:16]}...')

In [ ]:
# data/knn_index is git-ignored, so a fresh session rebuilds it. It must be the same
# index knn_fewshot retrieved from, hence base_qwen.yaml rather than the arm's config.
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
if not all((INDEX / f).exists() for f in INDEX_FILES):
    !python3 manage.py build_index --config configs/base_qwen.yaml

meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'], meta
INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest()[:16] for f in INDEX_FILES}
print(f"{meta['n_passages']} pool passages on {meta['embed_model']}, dim {meta['dim']}")
print(json.dumps(INDEX_SHA, indent=2))

---
## 4 — Routing, recorded per segment

In [ ]:
from src.retrieval.rarity import load_irregular
from src.retrieval.retrieve import RetrievalIndex
from src.retrieval.sparse import SparseRetriever

index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])
retriever = SparseRetriever(
    index, load_irregular(str(RARITY_PATH)), index,
    zwnj=RAR['zwnj'], m=SPA['m'], redundancy=SPA['redundancy'],
    min_query_terms=SPA['min_query_terms'],
)
SELECTED, TRACES = retriever.select_with_trace(SRC, k=RETR['k'])
print(f'{len(TRACES)} traces, {len(SELECTED[0])} exemplars per prompt')

In [ ]:
SLOTS = min(SPA['m'], RETR['k'])
HIST = {str(v): sum(t['n_sparse'] == v for t in TRACES) for v in range(SLOTS + 1)}
ROUTES = {r: sum(t['route'] == r for t in TRACES) for r in ('full', 'partial', 'dense')}

diag = json.loads(DIAG.read_text(encoding='utf-8'))
assert diag['config']['index_dir'] == RETR['index_dir'], diag['config']
assert HIST == diag['n_sparse']['histogram'], f'{HIST} against {diag["n_sparse"]["histogram"]}'
assert ROUTES == diag['routes'], f'{ROUTES} against {diag["routes"]}'

ROUTING.write_text(json.dumps({
    'split': SPLIT,
    'index_dir': RETR['index_dir'],
    'rarity_sha256': RARITY_SHA,
    'k': RETR['k'],
    'm': SPA['m'],
    'routes': ROUTES,
    'n_sparse_histogram': HIST,
    'segments': [{'route': t['route'], 'n_sparse': t['n_sparse'], 'coverage': t['coverage'],
                  'n_query_terms': t['n_query_terms']} for t in TRACES],
}, ensure_ascii=False, indent=2), encoding='utf-8')

routed = len(TRACES) - ROUTES['dense']
print(f'routes {ROUTES}  ({routed / len(TRACES):.1%} routed)')
print(f'rarity slots filled {HIST}, mean {np.mean([t["n_sparse"] for t in TRACES]):.3f}')
print(f'matches {DIAG}; wrote {ROUTING}')

---
## 5 — Generation

In [ ]:
t0 = time.perf_counter()
r = subprocess.run([PY, 'manage.py', 'infer', '--condition', ARM, '--config', str(CONFIG)],
                   check=False)
assert r.returncode == 0, f'{ARM} exited {r.returncode}'
GEN_SECONDS = round(time.perf_counter() - t0, 1)
print(f'{GEN_SECONDS / 60:.1f} min, finished {datetime.now(timezone.utc).isoformat()}')

---
## 6 — The output and its provenance

In [ ]:
ARM_PATH = OUT / f'{ARM}_{SPLIT}.jsonl'
ARM_ROWS = [json.loads(x) for x in ARM_PATH.open(encoding='utf-8') if x.strip()]

assert len(ARM_ROWS) == len(VAL), f'{len(ARM_ROWS)} rows, expected {len(VAL)}'
assert [r['input'] for r in ARM_ROWS] == SRC, 'source order differs from the eval file'
assert all(r['condition'] == ARM for r in ARM_ROWS), 'mislabelled rows'
blank = [i for i, r in enumerate(ARM_ROWS) if not r['prediction'].strip()]
errored = [i for i, r in enumerate(ARM_ROWS) if 'error' in r]
assert not errored, f'{len(errored)} segments recorded an error: {errored[:5]}'
print(f'{len(ARM_ROWS)} rows, {len(blank)} blank predictions, {len(errored)} errors')

In [ ]:
USAGE = json.loads((OUT / f'{ARM}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
PROV = USAGE['provenance']

assert PROV['index_dir'] == RETR['index_dir'], PROV['index_dir']
assert PROV['rarity_sha256'] == RARITY_SHA, f"{PROV['rarity_sha256']} != {RARITY_SHA}"
assert PROV['k'] == RETR['k'] and PROV['m'] == SPA['m'], PROV
assert (PROV['df_min'], PROV['df_max']) == (RAR['df_min'], RAR['df_max']), PROV
assert PROV['min_query_terms'] == SPA['min_query_terms'], PROV
assert PROV['redundancy'] == SPA['redundancy'], PROV
assert PROV['ordering'] == PROMPT['ordering'], PROV
print(json.dumps(PROV, indent=2))
print(f"{USAGE['calls']} calls, ${USAGE.get('cost_usd', 0.0):.2f} — local weights, nothing paid")

---
## 7 — Divergence from knn_fewshot

In [ ]:
ARM_PRED = [r['prediction'] for r in ARM_ROWS]
BASE_PRED = [r['prediction'] for r in BASE_ROWS]
DIFFERS = np.array([a != b for a, b in zip(ARM_PRED, BASE_PRED)])
ROUTE = np.array([t['route'] for t in TRACES])
N_SPARSE = np.array([t['n_sparse'] for t in TRACES])

share = DIFFERS.mean()
print(f'{DIFFERS.sum()}/{len(DIFFERS)} predictions differ ({share:.1%}); '
      f'routed fraction is {routed / len(TRACES):.1%}')
for r in ('full', 'partial', 'dense'):
    sel = ROUTE == r
    print(f'  {r:8s} n={sel.sum():5d}  differ {DIFFERS[sel].mean():.1%}')

In [ ]:
# Dense-routed prompts are byte-identical to the baseline's, so a difference there is
# decode nondeterminism rather than selection, and bounds the noise on the other two rows.
dense_diff = DIFFERS[ROUTE == 'dense'].mean()
assert 0.75 <= share <= 0.98, (
    f'{share:.1%} of predictions differ against a {routed / len(TRACES):.1%} routed fraction; '
    f'selection is not what moved the outputs')
if dense_diff:
    print(f'WARNING: {dense_diff:.1%} of dense-routed segments differ despite an identical '
          f'prompt — greedy decoding did not reproduce across sessions, so read the deltas '
          f'below against this floor, not against zero.')
else:
    print('every dense-routed segment reproduces the baseline exactly; the deltas below '
          'carry no decode noise')

In [ ]:
i = int(np.flatnonzero((ROUTE == 'full') & DIFFERS)[0])
print('SOURCE  :', SRC[i][:110])
print('terms   :', TRACES[i]['query_terms'][:8], f"coverage {TRACES[i]['coverage']}")
print(f'{BASELINE:12s}:', BASE_PRED[i][:200])
print(f'{ARM:12s}:', ARM_PRED[i][:200])

---
## 8 — Stage D: scoring

In [ ]:
CONDS = [BASELINE, ARM]
!{PY} manage.py eval --conditions {' '.join(CONDS)} --split {SPLIT}

In [ ]:
from src.eval.quick import score

SURFACE = {c: score(c, OUT, SPLIT) for c in CONDS}
print(f"{'condition':14s} {'chrF':>8s} {'BLEU':>8s} {'markers/seg':>12s}")
for cond in CONDS:
    s = SURFACE[cond]
    print(f"{cond:14s} {s['chrF']:8.2f} {s['BLEU']:8.2f} {s['marker_rate']:12.2f}")
print(f"gold targets carry {SURFACE[BASELINE]['ref_marker_rate']:.2f} markers per segment")

In [ ]:
if not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install', '-q']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)

COMET_PATH = f'results/comet_{SPLIT}.json'
PRIOR_COMET = set(json.loads(Path(COMET_PATH).read_text(encoding='utf-8')))
assert BASELINE in PRIOR_COMET, f'{BASELINE} is not in {COMET_PATH} to pair against'

r = subprocess.run([COMET_PY, 'manage.py', 'comet', '--conditions', ARM, '--split', SPLIT,
                    '--results_path', COMET_PATH, '--batch_size', '16'], check=False)
assert r.returncode == 0, f'comet exited {r.returncode}'

In [ ]:
COMET = json.loads(Path(COMET_PATH).read_text(encoding='utf-8'))
assert PRIOR_COMET <= set(COMET), f'lost from {COMET_PATH}: {sorted(PRIOR_COMET - set(COMET))}'
ref, arm = COMET[BASELINE], COMET[ARM]
assert arm['n'] == len(VAL), arm['n']
assert arm['model'] == ref['model'], (arm['model'], ref['model'])
assert arm['sources'] == ref['sources'], 'the two conditions are not paired segment for segment'
print(f"{ref['model']}: {BASELINE} {ref['system']:.4f} -> {ARM} {arm['system']:.4f}")

### Register fit

In [ ]:
!{PY} manage.py stylometrics --conditions {' '.join(CONDS)} --split {SPLIT} --targets-split train

In [ ]:
STYLO_PATH = f'results/stylometrics_ci_{ARM}_{SPLIT}.json'
!{PY} manage.py stylometrics_ci --split {SPLIT} --conditions {' '.join(CONDS)} \
    --n_resamples {N_STYLO} --alpha {ALPHA} --seed {SEED} --results_path {STYLO_PATH}

In [ ]:
STYLO = json.loads(Path(STYLO_PATH).read_text(encoding='utf-8'))
old = json.loads(Path(f'results/stylometrics_ci_{SPLIT}.json').read_text(encoding='utf-8'))
a, b = STYLO['cells'][BASELINE], old['cells'][BASELINE]
assert a['stylo_dist'] == b['stylo_dist'] and a['z'] == b['z'], f'{BASELINE} moved between passes'
print(f"{BASELINE} reproduces the committed row: stylo_dist {a['stylo_dist']:.4f}")
for cond in CONDS:
    print(f"  {cond:14s} stylo_dist {STYLO['cells'][cond]['stylo_dist']:.4f}")

---
## 9 — Phi (paid)

In [ ]:
JUDGE_CFG = 'configs/judge_eval.yaml'
JUDGE_RESULTS = f'results/judge_{SPLIT}.json'
JUDGE_USAGE = f'results/judge_{SPLIT}_usage.json'
JUDGE_CI_PATH = f'results/judge_ci_{ARM}_{SPLIT}.json'

PRIOR_JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
PRIOR_USAGE = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
PRIOR_SPEND, PRIOR_CALLS = (PRIOR_USAGE['cumulative'][k] for k in ('cost_usd', 'calls'))

assert BASELINE in PRIOR_JUDGE, f'{BASELINE} has no Phi to compare against'
BUY = [c for c in CONDS if c not in PRIOR_JUDGE]
PER_CALL = PRIOR_USAGE['cumulative']['cost_usd'] / PRIOR_USAGE['cumulative']['calls']
N_CALLS = len(VAL) * len(BUY)
PROJECTED = PER_CALL * N_CALLS
print(f"buying Phi for {BUY or 'nothing'}: {N_CALLS} calls at ${PER_CALL:.5f} "
      f"= ${PROJECTED:.2f} projected (cumulative judge spend ${PRIOR_SPEND:.2f})")

In [ ]:
# Left False so a top-to-bottom re-run cannot authorise itself.
SPEND_OK = False
BUDGET_USD = 1.80
N_PILOT = 25

assert PROJECTED <= BUDGET_USD, f'projection ${PROJECTED:.2f} exceeds the ${BUDGET_USD:.2f} cap'
print(f'authorised {SPEND_OK}   cap ${BUDGET_USD:.2f}   pilot {N_PILOT} x {len(BUY)} '
      f'(${PER_CALL * N_PILOT * len(BUY):.3f})')

In [ ]:
if not BUY:
    print('nothing to buy: every condition already carries Phi')
else:
    assert SPEND_OK, 'set SPEND_OK = True in the cell above to authorise the pilot'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG, '--limit', str(N_PILOT)], check=False)
    assert r.returncode == 0, f'pilot exited {r.returncode}'

In [ ]:
_u = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
if _u['conditions'] != sorted(BUY) or _u['limit'] != N_PILOT:
    REVISED = PROJECTED
    print(f"{JUDGE_USAGE} holds {_u['conditions']} at limit {_u['limit']}, not this pilot: "
          f'${PROJECTED:.2f} stands')
else:
    pilot = _u['session']
    REVISED = pilot['cost_usd'] / pilot['calls'] * N_CALLS
    print(f"pilot {pilot['calls']} calls at ${pilot['cost_usd'] / pilot['calls']:.5f} "
          f'-> ${REVISED:.2f} for the full pass')
assert REVISED <= BUDGET_USD, f'revised ${REVISED:.2f} exceeds the ${BUDGET_USD:.2f} cap'

In [ ]:
# The client is built on first call, so re-running over a complete cache makes no request.
if not BUY:
    print('nothing to buy; the results file already carries every condition')
else:
    assert SPEND_OK, 'set SPEND_OK = True to authorise the full pass'
    r = subprocess.run([PY, 'manage.py', 'judge', '--conditions', *BUY, '--split', SPLIT,
                        '--config', JUDGE_CFG], check=False)
    assert r.returncode == 0, f'judge exited {r.returncode}'

In [ ]:
JUDGE = json.loads(Path(JUDGE_RESULTS).read_text(encoding='utf-8'))
HAVE_PHI = ARM in JUDGE
if HAVE_PHI:
    r = subprocess.run([PY, 'manage.py', 'judge_ci', '--split', SPLIT, '--conditions', *CONDS,
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--results_path', JUDGE_CI_PATH], check=False)
    assert r.returncode == 0, f'judge_ci exited {r.returncode}'
else:
    print(f'{ARM} carries no Phi; sections 10 and 11 report adequacy and register only')

In [ ]:
lost = sorted(set(PRIOR_JUDGE) - set(JUDGE))
assert not lost, f'lost from {JUDGE_RESULTS}: {lost}'
if HAVE_PHI:
    assert JUDGE[ARM]['model'] == JUDGE[BASELINE]['model'], 'two raters, not one'

usage = json.loads(Path(JUDGE_USAGE).read_text(encoding='utf-8'))
assert usage['priced'], 'the judge model has no pricing table; cost_usd is a floor, not a bill'
SPENT = usage['cumulative']['cost_usd'] - PRIOR_SPEND
PAID_CALLS = usage['cumulative']['calls'] - PRIOR_CALLS
assert SPENT <= BUDGET_USD, f'${SPENT:.2f} spent against a ${BUDGET_USD:.2f} cap'

METRICS = ['chrf', 'bleu', 'comet'] + (['judge'] if HAVE_PHI else [])
print(f"{PAID_CALLS} paid calls, ${SPENT:.2f} on {usage['model']} this session; "
      f"cumulative ${usage['cumulative']['cost_usd']:.2f}")
print('reading out on', ', '.join(METRICS))

---
## 10 — The paired bootstrap

In [ ]:
BOOT_PATHS = {}
for metric in METRICS:
    path = f'results/bootstrap_{metric}_{ARM}_{SPLIT}.json'
    r = subprocess.run([PY, 'manage.py', 'bootstrap', '--metric', metric,
                        '--conditions', *CONDS, '--split', SPLIT, '--pairs', f'{ARM}:{BASELINE}',
                        '--n_resamples', str(N_BOOT), '--alpha', str(ALPHA), '--seed', str(SEED),
                        '--out', path], check=False)
    assert r.returncode == 0, f'{metric} bootstrap exited {r.returncode}'
    BOOT_PATHS[metric] = path

---
## 11 — Read-out by dose

In [ ]:
from src.eval.bootstrap import _load_segment_scores, paired_bootstrap

ROUTE_JSON = json.loads(ROUTING.read_text(encoding='utf-8'))['segments']
assert len(ROUTE_JSON) == len(VAL), len(ROUTE_JSON)

STRATA = {
    f'full dose (n_sparse = {SLOTS})': [i for i, t in enumerate(ROUTE_JSON)
                                        if t['n_sparse'] == SLOTS],
    'all routed': [i for i, t in enumerate(ROUTE_JSON) if t['route'] != 'dense'],
    'all segments': list(range(len(VAL))),
}
# Declared at n=1,323; scaled by 1/sqrt(n) for the subsets.
FLOOR = {'judge': 0.058, 'comet': 0.005}
print({k: len(v) for k, v in STRATA.items()})

In [ ]:
SCORES = {}
for metric in METRICS:
    scores, sources = _load_segment_scores(metric, CONDS, OUT, SPLIT, None)
    for cond in CONDS:
        assert cond in scores, f'{metric}: {cond} has no per-segment scores'
        assert len(scores[cond]) == len(VAL), (metric, cond, len(scores[cond]))
        if sources.get(cond) is not None:
            assert sources[cond] == SRC, f'{metric}/{cond}: segment order is not the eval order'
    SCORES[metric] = scores
print('per-segment scores aligned to the eval order for', ', '.join(SCORES))

In [ ]:
PLACES = {'chrf': 2, 'bleu': 2, 'comet': 4, 'judge': 4}

def delta(metric, idx):
    a = [SCORES[metric][ARM][i] for i in idx]
    b = [SCORES[metric][BASELINE][i] for i in idx]
    return paired_bootstrap(a, b, n_resamples=N_BOOT, alpha=ALPHA, seed=SEED)

for metric in METRICS:
    p = PLACES[metric]
    print(f'\n{metric}   {ARM} - {BASELINE}')
    for label, idx in STRATA.items():
        d = delta(metric, idx)
        mark = '*' if d['significant'] else ' '
        line = (f"  {label:26s} n={d['n']:5d}  {d['diff']:+.{p}f} "
                f"[{d['ci_low']:+.{p}f}, {d['ci_high']:+.{p}f}]  p={d['p_value']:.4f} {mark}")
        if metric in FLOOR:
            f = FLOOR[metric] * (len(VAL) / d['n']) ** 0.5
            line += f"  floor {f:.{p}f}{'' if abs(d['diff']) >= f else '  (under)'}"
        print(line)

In [ ]:
# The CLI's own table, for the pair the command line names.
for metric, path in BOOT_PATHS.items():
    rec = next(r for r in json.loads(Path(path).read_text(encoding='utf-8'))['comparisons']
               if (r['a'], r['b']) == (ARM, BASELINE))
    p = PLACES[metric]
    print(f"{metric:6s} {rec['diff']:+.{p}f} [{rec['ci_low']:+.{p}f}, {rec['ci_high']:+.{p}f}] "
          f"p={rec['p_value']:.4f} n={rec['n']}")

---
## 12 — Seal

In [ ]:
assert not list(OUT.glob('*_test.jsonl')), 'a test-split output exists'
assert not list(Path('results').glob('*_test.json')), 'a test-split result exists'
assert USAGE.get('cost_usd', 0.0) == 0.0, USAGE
print(f'generation: {USAGE["calls"]} calls, $0.00 (local weights)')
print(f'judge: {PAID_CALLS} paid calls, ${SPENT:.2f}')

In [ ]:
ARTEFACTS = ([str(ARM_PATH), str(OUT / f'{ARM}_{SPLIT}_usage.json'), str(ROUTING),
              COMET_PATH, STYLO_PATH, *BOOT_PATHS.values()]
             + ([JUDGE_RESULTS, JUDGE_CI_PATH] if HAVE_PHI else []))
for f in ARTEFACTS:
    assert Path(f).exists(), f
!git status --short {' '.join(ARTEFACTS)}

In [ ]:
import zipfile

BUNDLE = Path(f'{ARM}_{SPLIT}.zip')
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ARTEFACTS:
        z.write(f, f)
print(f'{BUNDLE}  {BUNDLE.stat().st_size / 1e6:.1f} MB')